In [3]:
import model
import data

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

## Load Dataset

In [4]:
train_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_train.npy", ks_npy="../Data/ks_train.npy")
val_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_val.npy", ks_npy="../Data/ks_val.npy")
test_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_test.npy", ks_npy="../Data/ks_test.npy")

# train_dataset = train_dataset.to(device)
# val_dataset = val_dataset.to(device)
# test_dataset = test_dataset.to(device)

train_loader = DataLoader(train_dataset, batch_size=65536, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=65536, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=65536, shuffle=False)

## Model instantiation

In [7]:
m = model.MLP([10, 64, 64, 64, 6])

m = m.to(device)

criterion = nn.MSELoss()
optim = torch.optim.Adam(m.parameters(), lr = 1e-4, weight_decay = 0.0001)

In [8]:
len(train_dataset)



9594400

## Training Loop

In [ ]:
from tqdm import tqdm
EPOCHS = 1000

prev_val_loss = float("inf")
no_improvement_counter = 0
patience = 3

train_loss_history = []
val_loss_history = []

for e in range(EPOCHS):
    loss_accum = 0
    for i, o in tqdm(train_loader):
        i = i.to(device)
        o = o.to(device)


        pred = m(i)

        loss = criterion(pred, o)
        loss.backward()
        optim.step()

        optim.zero_grad()

        loss_accum += loss.item() 

    torch.save(m.state_dict(), f"./models/latest_model.pth")

    train_loss_history.append(loss_accum/len(train_loader))
    print(f"Epoch {e} Loss: {loss_accum/len(train_loader)}")
    if e % 1 == 0:
        with torch.no_grad():
            val_loss_accum = 0
            for i, o in tqdm(val_loader):
                i = i.to(device)
                o = o.to(device)

                pred = m(i)

                loss = criterion(pred, o)
                val_loss_accum += loss.item() 

        print(f"Epoch {e} Val Loss: {val_loss_accum/len(val_loader)}")
        val_loss_history.append(val_loss_accum/len(val_loader)) 
        if val_loss_accum/len(val_loader) < prev_val_loss:
            prev_val_loss = val_loss_accum/len(val_loader)
            no_improvement_counter = 0
            torch.save(m.state_dict(), f"./models/best_model.pth")
        else:
            no_improvement_counter += 1

        if no_improvement_counter >= patience:
            print("Early stopping triggered.")
            break




100%|██████████| 147/147 [10:25<00:00,  4.26s/it]


Epoch 0 Loss: 0.09304607304788771


100%|██████████| 19/19 [01:13<00:00,  3.87s/it]


Epoch 0 Val Loss: 0.07592631670597352


  3%|▎         | 4/147 [00:17<10:20,  4.34s/it]